In [1]:
# ── 0. 라이브러리 불러오기 ─────────────────────────────────────────
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, MinMaxScaler

pd.set_option("display.max_columns", None)  # 컬럼이 많아도 잘리지 않고 다 보이게 설정


In [2]:
# ── 0-1. 데이터 불러오기 (df_origin: 원본 보존용) ────────────────────
CSV_PATH = "Flight_Price.csv"  # ← 실제 파일명/경로로 바꿔서 쓰세요 (예: 'Clean_Dataset.csv')

try:
    df_origin = pd.read_csv(CSV_PATH)
    print(f"✅ '{CSV_PATH}' 파일을 불러왔습니다. shape={df_origin.shape}")
except FileNotFoundError:
    # 실습용 샘플 데이터 자동 생성 (실제 파일과 컬럼 구조는 동일)
    print(f"⚠️ '{CSV_PATH}'를 찾을 수 없어, 동일 구조의 샘플 데이터로 대신 실행합니다.")
    rng = np.random.default_rng(42)
    N = 200
    df_origin = pd.DataFrame({
        "airline": rng.choice(["SpiceJet", "AirAsia", "Vistara", "GO_FIRST", "Indigo", "Air_India"], N),
        "flight": [f"XX-{i}" for i in range(N)],
        "source_city": rng.choice(["Delhi", "Mumbai", "Bangalore"], N),
        "departure_time": rng.choice(["Evening", "Morning", "Early_Morning", "Night"], N),
        "stops": rng.choice(["zero", "one", "two_or_more"], N),
        "arrival_time": rng.choice(["Night", "Morning", "Early_Morning", "Afternoon"], N),
        "destination_city": rng.choice(["Mumbai", "Delhi", "Bangalore"], N),
        "class": rng.choice(["Economy", "Business"], N),
        "duration": rng.uniform(1, 15, N).round(2),
        "days_left": rng.integers(1, 50, N),
        "price": rng.uniform(2000, 60000, N).round(0),
    })

df_origin.head()


⚠️ 'Flight_Price.csv'를 찾을 수 없어, 동일 구조의 샘플 데이터로 대신 실행합니다.


,airline,flight,source_city,departure_time,stops,arrival_time,destination_city,class,duration,days_left,price
0,SpiceJet,XX-0,Mumbai,Night,one,Night,Bangalore,Business,4.38,3,5600.0
1,Indigo,XX-1,Bangalore,Night,two_or_more,Morning,Bangalore,Economy,5.51,23,28579.0
2,GO_FIRST,XX-2,Mumbai,Morning,two_or_more,Early_Morning,Delhi,Business,3.18,23,9484.0
3,Vistara,XX-3,Bangalore,Night,zero,Afternoon,Bangalore,Business,13.24,13,10835.0
4,Vistara,XX-4,Mumbai,Morning,two_or_more,Early_Morning,Mumbai,Business,4.97,7,38672.0


In [3]:
# =====================================================================
# SECTION 02. 범주형 데이터 정제하기 (인코딩)
# =====================================================================

# ── 1. 레이블 인코딩하기 ────────────────────────────────────────────

# 1) 판다스에서 레이블 인코딩하기 (pd.factorize) --------------------
# 데이터 구간화(인코딩) 전 원본 데이터 불러오기
df = df_origin.copy()

# factorize(): 범주형 데이터를 "등장 순서" 기준으로 정수 인코딩
# 반환값 (codes, uniques) 중 [0]인 codes만 사용, reshape(-1,1)로 새 컬럼에 대입 가능한 형태로 변환
df["label_encoding"] = pd.factorize(df["airline"])[0].reshape(-1, 1)
print("\n[1-1] pandas factorize 결과")
print(df[["airline", "label_encoding"]].head())

# airline 컬럼과 새롭게 만들어진 label_encoding 컬럼의 빈도표 확인하기
print(df["airline"].value_counts())
print(df["label_encoding"].value_counts())



[1-1] pandas factorize 결과
    airline  label_encoding
0  SpiceJet               0
1    Indigo               1
2  GO_FIRST               2
3   Vistara               3
4   Vistara               3
airline
Indigo       45
Vistara      41
SpiceJet     33
GO_FIRST     31
Air_India    25
AirAsia      25
Name: count, dtype: int64
label_encoding
1    45
3    41
0    33
2    31
4    25
5    25
Name: count, dtype: int64


In [ ]:
# 2) 사이킷런으로 레이블 인코딩하기 (LabelEncoder) -------------------
# 사이킷런 패키지의 LabelEncoder 불러오기 (파일 맨 위에서 이미 import 완료)

# LabelEncoder로 airline 컬럼 레이블 인코딩하기
le = LabelEncoder()
df["airline_Label_Encoder"] = le.fit_transform(df["airline"])
print("\n[1-2] sklearn LabelEncoder 결과 (factorize와 비교)")
print(df[["airline", "label_encoding", "airline_Label_Encoder"]].head())

# airline 컬럼과 새롭게 만들어진 airline_Label_Encoder 컬럼의 빈도표 확인하기
print(df["airline"].value_counts())
print(df["airline_Label_Encoder"].value_counts())

# 레이블 인코딩 역변환(디코딩)하기
decoded = le.inverse_transform(df["airline_Label_Encoder"]).reshape(-1, 1)
print("\n[1-3] 역변환(디코딩) 결과 (앞 5개)")
print(decoded[:5])

In [ ]:
# ── 2. 원핫 인코딩하기 ──────────────────────────────────────────────

# 1) 판다스에서 원핫 인코딩하기 (pd.get_dummies) ---------------------
# 레이블 인코딩 전 원본 데이터 불러오기
df = df_origin.copy()

# class 컬럼을 원핫 인코딩하기 (미리보기)
print("\n[2-1] pd.get_dummies 미리보기")
print(pd.get_dummies(df["class"]).head())

# 원핫 인코딩 결과를 데이터에 반영하기
df = pd.get_dummies(df, columns=["class"])
print(df.head())

In [ ]:
# 2) 사이킷런으로 원핫 인코딩하기 (OneHotEncoder) --------------------
# 판다스 원핫 인코딩 전 원본 데이터 다시 불러오기
df = df_origin.copy()

# OneHotEncoder로 원핫 인코딩하기
oh = OneHotEncoder()
encoder = oh.fit_transform(df["class"].values.reshape(-1, 1)).toarray()

# 원핫 인코딩 결과를 데이터프레임으로 만들기
df_OneHot = pd.DataFrame(
    encoder,
    columns=["class_" + str(oh.categories_[0][i]) for i in range(len(oh.categories_[0]))]
)

# 원핫 인코딩 결과를 원본 데이터에 붙여넣기
df1 = pd.concat([df, df_OneHot], axis=1)
print("\n[2-2] sklearn OneHotEncoder 결과")
print(df1.head())